# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:

# Task 1 — World Choropleth: Life Expectancy Diverging Scale

import pandas as pd
import plotly.express as px

# Step 1: Load Gapminder dataset
df = px.data.gapminder()

# Step 2: Filter data for 2007
gm_2007 = df[df["year"] == 2007].copy()

# Step 3: Calculate global average life expectancy
global_avg = gm_2007["lifeExp"].mean()

# Step 4: Calculate deviation from global average
gm_2007["deviation"] = gm_2007["lifeExp"] - global_avg

# Step 5: Find the continent furthest below average
continent_avg = gm_2007.groupby("continent")["lifeExp"].mean()
lowest_continent = continent_avg.idxmin()

# Step 6: Create world choropleth map
fig = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    locationmode="ISO-3",
    color="deviation",
    hover_name="country",
    hover_data={
        "lifeExp": ":.1f",
        "deviation": ":+.1f",
        "iso_alpha": False
    },
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title=f"{lowest_continent} Had the Lowest Average Life Expectancy in 2007",
    labels={
        "deviation": "Years from global average",
        "lifeExp": "Life expectancy (years)"
    },
    projection="natural earth"
)

# Step 7: Format the map
fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor="lightgray",
        showland=True,
        landcolor="#F2F2F2"
    ),
    coloraxis_colorbar=dict(
        title="Deviation<br>(years)"
    ),
    height=600
)

# Step 8: Display the map
fig.show()

# Print summary
print(f"Global average life expectancy (2007): {global_avg:.2f} years")
print(f"Continent furthest below average: {lowest_continent}")
print(f"Average life expectancy in {lowest_continent}: {continent_avg.min():.2f} years")


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here*

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here*


In [ ]:
# Task 2
# YOUR CODE HERE

# Load your GeoJSON
# with open('your_file.geojson', 'r') as file:
#     my_geojson = json.load(file)

# Inspect the properties to find the right featureidkey
# print(my_geojson['features'][0]['properties'])

# Task 2 — Custom GeoJSON Choropleth Map

import pandas as pd
import plotly.express as px
import requests

# Step 1: Load Gapminder data for 2007
df = px.data.gapminder()
gm_2007 = df[df["year"] == 2007].copy()

# Step 2: Download an external GeoJSON file
geojson_url = (
    "https://raw.githubusercontent.com/datasets/"
    "geo-countries/master/data/countries.geojson"
)

response = requests.get(geojson_url, timeout=30)
response.raise_for_status()
my_geojson = response.json()

# Step 3: Inspect the GeoJSON properties
print("GeoJSON properties:")
print(my_geojson["features"][0]["properties"])

# Step 4: Match GeoJSON country codes to Gapminder
# The geo-countries dataset uses ISO3166-1-Alpha-3
feature_key = "properties.ISO3166-1-Alpha-3"

geo_codes = {
    feature["properties"].get("ISO3166-1-Alpha-3")
    for feature in my_geojson["features"]
}

gm_2007["matched"] = gm_2007["iso_alpha"].isin(geo_codes)

print("Matched countries:", gm_2007["matched"].sum())
print("Total Gapminder countries:", len(gm_2007))

# Step 5: Create the choropleth
fig2 = px.choropleth(
    gm_2007,
    geojson=my_geojson,
    locations="iso_alpha",
    featureidkey=feature_key,
    color="lifeExp",
    hover_name="country",
    hover_data={
        "lifeExp": ":.1f",
        "continent": True,
        "iso_alpha": False,
        "matched": False
    },
    color_continuous_scale="Viridis",
    range_color=(35, 85),
    title="Europe Outpaced Much of Sub-Saharan Africa in Life Expectancy (2007)",
    labels={
        "lifeExp": "Life expectancy (years)",
        "continent": "Continent"
    },
    projection="natural earth"
)

# Step 6: Format the map
fig2.update_geos(
    fitbounds="locations",
    visible=False,
    showcoastlines=True,
    coastlinecolor="lightgray"
)

fig2.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=650,
    margin=dict(l=20, r=20, t=80, b=20),
    coloraxis_colorbar=dict(
        title="Life expectancy<br>(years)"
    )
)

# Step 7: Display the map
fig2.show()
